# Monarch · 36s film renderer (Wan 2.1, free Colab GPU)

**2 tap:** (1) `Runtime → Change runtime type → T4 GPU` → Save, (2) `Runtime → Run all`.
Phir chai pee lo. Clips khud Google Drive me `Monarch-T2V/clips/` me jati rahengi (resume-safe:
agar session toot jaye to dobara Run all — jo ban chuka hai woh skip ho jayega).

Wan 2.1 T2V-1.3B = 2025 ka asli text-to-video model. Har scene ke har frame model banata hai —
koi photo/slideshow nahi.


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU chuno, phir Run all"
p = torch.cuda.get_device_properties(0)
print("VRAM:", round(p.total_memory / 1e9, 1), "GB")


In [ ]:
%pip -q install "diffusers>=0.32" "transformers>=4.49" accelerate safetensors sentencepiece protobuf ftfy imageio imageio-ffmpeg
import diffusers, transformers
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)


In [ ]:
import pathlib
OUT = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT = pathlib.Path('/content/drive/MyDrive/Monarch-T2V/clips')
    OUT.mkdir(parents=True, exist_ok=True)
    print("Drive folder ready:", OUT)
except Exception as exc:
    print("Drive mount nahi hua:", type(exc).__name__, exc)
    print("Aap Drive mount approve karke yeh cell dobara chala dein.")


In [ ]:
import pathlib
if OUT is None:                      # Drive na ho to /content me
    OUT = pathlib.Path('/content/clips'); OUT.mkdir(exist_ok=True)
print("clips yahan jayengi:", OUT)

JOB = {
 "id": "personality-wan",
 "model": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
 "style": "photorealistic, shot on 35mm film, Kodak Portra 400, natural skin texture, available light, shallow depth of field, realistic colours, documentary camera, no CGI, no illustration",
 "negative": "blurry, low quality, watermark, text, static, overexposed, cartoon, anime, deformed face, extra limbs, plastic skin, cgi",
 "width": 480,
 "height": 832,
 "frames": 49,
 "raw_fps": 16,
 "guidance": 5.0,
 "scenes": [
  {
   "id": "s01",
   "seed": 101,
   "prompt": "a South Asian man in his thirties in a dark wool coat walks slowly toward the camera on a wet city street at night, rain falling past him, neon signs reflected in the asphalt, headlights sliding by behind, handheld camera"
  },
  {
   "id": "s02",
   "seed": 102,
   "prompt": "rain water running down a dark window at night, city lights blurred behind the glass, droplets joining and sliding, faint reflection of a face in the pane, slow rack focus"
  },
  {
   "id": "s03",
   "seed": 103,
   "prompt": "two men sitting across a small cafe table at night, warm lamp light, the listener leans in and nods slowly, steam rising from a cup, background out of focus, handheld"
  },
  {
   "id": "s04",
   "seed": 104,
   "prompt": "over the shoulder shot of a man speaking quietly at a table, his hands moving slightly as he talks, warm tungsten light on his cheek, smoke drifting in the background, shallow focus"
  },
  {
   "id": "s05",
   "seed": 105,
   "prompt": "medium close up of a man looking straight into the camera, he blinks slowly, small catchlights in his eyes, dark room lit by a single lamp from the side, natural skin, slow push in"
  },
  {
   "id": "s06",
   "seed": 106,
   "prompt": "a man standing with his shoulders back in front of a tall window at dusk, he takes a slow breath and steadies himself, dust drifting through a shaft of light, camera drifts sideways"
  },
  {
   "id": "s07",
   "seed": 107,
   "prompt": "a night meeting room, one man sits still at a long table while blurred colleagues argue around him, fluorescent light flickers once, papers on the table, slow camera orbit"
  },
  {
   "id": "s08",
   "seed": 108,
   "prompt": "three friends laughing around a kitchen table at night, warm tungsten light, one of them leans back and laughs, a cup being lifted, subtle handheld movement, candid"
  },
  {
   "id": "s09",
   "seed": 109,
   "prompt": "a man alone in an empty room at dawn practising speaking to himself calmly, pale morning light through a window, he gestures once with his hand, dust in the air, static camera"
  },
  {
   "id": "s10",
   "seed": 110,
   "prompt": "silhouette of a man standing at a rooftop railing at sunrise, city haze below, his coat moving in the wind, camera slowly rising behind him"
  },
  {
   "id": "s11",
   "seed": 111,
   "prompt": "a man in a dark coat walks steadily through a bright hotel lobby, people blurred on both sides, marble floor reflecting light, camera tracks backwards ahead of him"
  },
  {
   "id": "s12",
   "seed": 112,
   "prompt": "close up of a vintage chrome microphone in a dark studio, dust drifting in a beam of light, the microphone cable swaying slightly, slow drift, deep blacks"
  }
 ]
}
print(len(JOB["scenes"]), "scenes |", JOB["width"], "x", JOB["height"], "|", JOB["frames"], "frames")


In [ ]:
import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video

pipe = WanPipeline.from_pretrained(JOB["model"], torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()      # T4 (16 GB) par fit
pipe.vae.to(torch.float32)           # fp16 VAE black frames deta hai
pipe.vae.enable_slicing()
pipe.set_progress_bar_config(disable=True)
print("Wan ready")


In [ ]:
import time, json

STEPS = 24                # quality/waqt ka balance
results = []
todo = [s for s in JOB["scenes"]
        if not ((OUT / f'{s["id"]}.mp4').exists() and (OUT / f'{s["id"]}.mp4').stat().st_size > 50000)]
print(f"{len(todo)} scenes baqi hain (baqi pehle se ban chuki hain)")

for n, scene in enumerate(todo, 1):
    t0 = time.time()
    print(f'[{n}/{len(todo)}] {scene["id"]} render ho rahi hai...', flush=True)
    try:
        with torch.no_grad():
            frames = pipe(
                prompt=f'{scene["prompt"]}, {JOB["style"]}',
                negative_prompt=JOB["negative"],
                height=JOB["height"], width=JOB["width"],
                num_frames=JOB["frames"], num_inference_steps=STEPS,
                guidance_scale=JOB["guidance"],
                generator=torch.Generator("cpu").manual_seed(int(scene["seed"])),
            ).frames[0]
        tmp = pathlib.Path('/content') / f'{scene["id"]}.mp4'
        export_to_video(frames, str(tmp), fps=JOB["raw_fps"])
        (OUT / f'{scene["id"]}.mp4').write_bytes(tmp.read_bytes())
        dt = round(time.time() - t0)
        print(f'{scene["id"]} OK in {dt // 60}m {dt % 60}s', flush=True)
        results.append({"scene": scene["id"], "state": "ok", "seconds": dt})
    except Exception as exc:
        print(f'{scene["id"]} FAIL {type(exc).__name__}: {str(exc)[:200]}', flush=True)
        results.append({"scene": scene["id"], "state": "fail", "error": f"{type(exc).__name__}: {str(exc)[:200]}"})
    (OUT / "receipt.json").write_text(json.dumps(results, indent=2))


In [ ]:
done = sorted(p.stem for p in OUT.glob('s*.mp4'))
print(f"{len(done)} / {len(JOB['scenes'])} clips ready:", ", ".join(done))
missing = [s["id"] for s in JOB["scenes"] if s["id"] not in done]
print("baqi:", missing if missing else "kuch nahi — poora set tayyar")
print("\nAb agent ko bolen: 'clips ready' — woh Drive se utha kar 36s film bana dega.")
